# CSP Name-Selection Screen v0 — recovered (NEWEST fork)

Recovered 2026-08-12 from the Colab Drive archive (see `wiki/colab-archive-audit.md`).
**It was missing from `tools/` entirely** despite being referenced in `wiki/trading-maxims.md:13`.

⛔ **CORRECTION, same day:** the first recovery grabbed fork `1srw5H5c` because it was the LARGEST
(57 KB). **Size was the wrong tiebreak — most of it was saved output.** Drive metadata (once the
connector came online) shows three forks modified within 11 minutes on 2026-07-17, and the LAST one
is `1DFo8dwo` at 20:20 — this file. The three have DIFFERENT code, so newest is the one that matters.

Enforces the trading-system laws AS FILTERS: Law 2 (200-day gate) · Law 3 (liquidity, market-cap floor,
optionable, no earnings within ~7 days) · Law 8 (every reject printed with the law that killed it).

**⚠️ The adverse-selection trap this screen was corrected for (`trading-maxims.md:13`): ranking by yield
reaches for the name the market prices for a reason.** A very high annualized yield is a WARNING, not a prize.
It proposes; Jake disposes. Nothing here places orders. Requires yfinance — **Colab run**.


# CSP Name-Selection Screen — v0 (the "which name" decision)

**What this is:** at one-put-at-a-time capital, *which name* is the entire decision. This screen finds
cash-secured-put candidates that pass the laws, then ranks the survivors by premium yield. It proposes —
**you decide.** Nothing here places orders.

**Laws it enforces as filters (from `CLAUDE.md`):**
- **Law 2 — the 200-day gate:** name must be trading ABOVE its 200-day SMA. Below = auto-reject.
- **Law 3 — name quality:** liquid (dollar-volume floor), optionable, market-cap floor, no earnings within
  ~7 days. The final "would I genuinely own it at the strike?" check is YOURS — a screen can't answer that.
- **Law 8 spirit — audit:** every reject is printed with the law/reason that killed it.

**How to run (iPhone / Colab):** open in Google Colab → Runtime → Run all. Token-free (yfinance, no keys).
Takes ~2–5 min depending on universe size.

**Honesty box:** quoted yields are *point-in-time mid-quotes* — off-hours quotes go stale and overstate the
mid. Yield is not expected return; the fat tail pays for the premium. A very high annualized yield is not a
prize, it's a WARNING (the market prices that name for a reason) — the screen flags those.

In [ ]:
# Cell 1 — setup (Colab-safe)
import sys, subprocess
try:
    import yfinance as yf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance", "pandas"])
    import yfinance as yf
import pandas as pd, numpy as np, math
from datetime import datetime, timedelta, timezone
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("yfinance", yf.__version__)

In [ ]:
# Cell 2 — CONFIG (edit these, rerun)
CAPITAL          = 1_000    # ring-fenced live target; collateral = strike x 100 must fit inside this
PRICE_MIN        = 4.00     # below this = penny-junk territory, auto-reject
DTE_MIN, DTE_MAX = 21, 45   # target expiration window (days to expiry)
DELTA_MIN, DELTA_MAX = 0.15, 0.35   # |put delta| band: ~15-35% assignment odds, the classic wheel zone
EARNINGS_BUFFER  = 7        # Law 3: no earnings within this many days
MIN_DOLLAR_VOL   = 15e6     # avg daily dollar volume floor (liquidity)
MIN_MARKET_CAP   = 1e9      # market-cap floor (not-junk)
MIN_OPEN_INT     = 100      # per-contract open-interest floor
MAX_SPREAD_PCT   = 0.25     # reject quotes where (ask-bid)/mid > this (fake/illiquid quotes)
RISK_FREE        = 0.04     # for the Black-Scholes delta approximation
YIELD_WARN       = 0.60     # annualized yield above this = "priced for a reason" flag, not a prize

In [ ]:
# Cell 3 — SEED UNIVERSE (a starting list, not an endorsement)
# Commonly optionable, historically lower-priced names. The FILTERS do the work — names that are now too
# expensive, illiquid, below the 200-day, or near earnings get rejected with a printed reason.
# ADD/REMOVE freely; this list is the only "manual" input in the screen.
UNIVERSE = sorted(set([
    # metals / materials / industrial
    "VALE", "CLF", "AA", "BTG", "KGC", "HMY", "AG", "PAAS", "UUUU", "UEC",
    # autos
    "F", "NIO", "LCID", "RIVN",
    # banks / financials / income names
    "BAC", "KEY", "RF", "HBAN", "SOFI", "NLY", "AGNC", "PSEC", "ITUB", "BBD",
    # telecom / media / consumer
    "T", "VZ", "NOK", "ERIC", "SIRI", "WBD", "SNAP", "PTON", "M", "KSS", "ABEV", "KVUE", "AMCR",
    # energy / coal / shipping
    "RIG", "ET", "KMI", "BTU", "ZIM",
    # travel
    "AAL", "CCL", "NCLH", "JBLU",
    # tech / growth / speculative (expect junk-flags and rejects here)
    "CHPT", "RUN", "PLUG", "JOBY", "ACHR", "DNA", "OPEN", "GRAB", "HOOD", "PLTR", "MARA", "RIOT", "TLRY",
]))
print(len(UNIVERSE), "tickers in seed universe")

In [ ]:
# Cell 4 — helpers
def bs_put_delta(S, K, T, sigma, r=RISK_FREE):
    """Black-Scholes put delta from the quoted IV. Approximation is fine — we only need the band."""
    if S <= 0 or K <= 0 or T <= 0 or sigma is None or sigma <= 0: return None
    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    N = lambda x: 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))
    return N(d1) - 1.0   # negative for puts

def next_earnings_days(tkr):
    """Days until next earnings, or None if unknown. Unknown != safe — we flag it."""
    try:
        cal = tkr.get_earnings_dates(limit=12)
        if cal is None or len(cal) == 0: return None
        now = pd.Timestamp.now(tz=cal.index.tz) if cal.index.tz else pd.Timestamp.now()
        future = [d for d in cal.index if d > now]
        if not future: return None
        return (min(future) - now).days
    except Exception:
        return None

rejects, passers, errors = [], [], []
def reject(sym, law, reason, **extra):
    rejects.append({"ticker": sym, "law": law, "reason": reason, **extra})

In [ ]:
# Cell 5 — THE SCREEN (per name: gates first, then the chain)
# v0.1: Law 4 is enforced on the STRIKE (K*100 <= CAPITAL), not the spot. The pre-filter only rejects
# names where even a ~20%-OTM strike couldn't fit — the chain decides the rest. v0 rejected on spot*100
# and threw away names like a $10.69 stock with a $10 strike ($1,000 collateral, fits exactly).
ladder = []   # law-4 rejects, kept as info: what a bigger capital box would buy

for sym in UNIVERSE:
    try:
        t = yf.Ticker(sym)
        h = t.history(period="1y", auto_adjust=True)
        if h is None or len(h) < 200:
            reject(sym, "-", "insufficient price history (<200 trading days)"); continue
        px    = float(h["Close"].iloc[-1])
        sma200 = float(h["Close"].rolling(200).mean().iloc[-1])
        dvol  = float((h["Close"] * h["Volume"]).tail(20).mean())

        # ---- affordability + junk floor ----
        if px < PRICE_MIN:
            reject(sym, "3", f"price ${px:.2f} < ${PRICE_MIN:.2f} floor (junk zone)"); continue
        if px * 100 * 0.80 > CAPITAL:   # even a ~20%-OTM strike can't fit -> truly unaffordable
            reject(sym, "4", f"price ${px:.2f}: even ~20%-OTM strike > ${CAPITAL:,} capital")
            ladder.append({"ticker": sym, "px": px, "collateral_at_spot_$": round(px * 100)}); continue

        # ---- Law 2: the 200-day gate ----
        if px <= sma200:
            reject(sym, "2", f"below 200-day (px ${px:.2f} <= sma ${sma200:.2f}) — STAND DOWN"); continue

        # ---- Law 3: liquidity, size, earnings ----
        if dvol < MIN_DOLLAR_VOL:
            reject(sym, "3", f"avg dollar volume ${dvol/1e6:.1f}M < ${MIN_DOLLAR_VOL/1e6:.0f}M floor"); continue
        try:
            mcap = t.fast_info.get("marketCap") or t.fast_info.get("market_cap")
        except Exception:
            mcap = None
        if mcap is not None and mcap < MIN_MARKET_CAP:
            reject(sym, "3", f"market cap ${mcap/1e9:.2f}B < ${MIN_MARKET_CAP/1e9:.0f}B floor"); continue
        edays = next_earnings_days(t)
        if edays is not None and edays <= EARNINGS_BUFFER:
            reject(sym, "3", f"earnings in {edays}d (buffer {EARNINGS_BUFFER}d)"); continue

        # ---- options: pick an expiration that does NOT hold through earnings (v0.2) ----
        # Law 3's point is dodging the earnings gap. A 21-45 DTE window spans earnings for ~half of
        # all names each quarter, so entry-day distance alone is not enough: the position must EXPIRE
        # at least EARNINGS_SETTLE_GAP days before the report (or earnings must be after expiration).
        EARNINGS_SETTLE_GAP = 2
        exps = t.options
        if not exps:
            reject(sym, "3", "no listed options"); continue
        today = datetime.now().date()
        cand = [(e, (datetime.strptime(e, "%Y-%m-%d").date() - today).days) for e in exps]
        cand = [(e, d) for e, d in cand if d >= 5]             # nothing shorter than ~a week
        if edays is not None:
            cand = [(e, d) for e, d in cand if d <= edays - EARNINGS_SETTLE_GAP]
        if not cand:
            reject(sym, "3", f"earnings in {edays}d: no expiration >=5 DTE settles before it"); continue
        in_window = [(e, d) for e, d in cand if DTE_MIN <= d <= DTE_MAX]
        short_dated = False
        if in_window:
            exp, dte = min(in_window, key=lambda x: abs(x[1] - (DTE_MIN + DTE_MAX) / 2))
        else:
            below = [(e, d) for e, d in cand if d < DTE_MIN]
            if not below:
                reject(sym, "3", f"no expiration fits (window {DTE_MIN}-{DTE_MAX}d, earnings in {edays}d)"); continue
            exp, dte = max(below, key=lambda x: x[1])          # longest one that still beats earnings
            short_dated = True

        puts = t.option_chain(exp).puts
        best = None
        # near-miss diagnostics: count WHY strikes fail so "no strike passed" is debuggable (Law 8 spirit)
        diag = {"otm_strikes": 0, "collateral": 0, "no_live_quote": 0, "wide_spread": 0,
                "low_oi": 0, "delta_out": 0}
        deltas_seen = []
        for _, row in puts.iterrows():
            K = float(row["strike"])
            if K >= px:            continue                    # OTM only
            diag["otm_strikes"] += 1
            if K * 100 > CAPITAL:                              # Laws 1/4: full cash collateral must fit
                diag["collateral"] += 1;  continue
            bid, ask = float(row.get("bid") or 0), float(row.get("ask") or 0)
            oi = float(row.get("openInterest") or 0)
            if bid <= 0 or ask <= 0:
                diag["no_live_quote"] += 1; continue           # no real market (or off-hours stale)
            mid = (bid + ask) / 2
            if (ask - bid) / mid > MAX_SPREAD_PCT:
                diag["wide_spread"] += 1;   continue
            if oi < MIN_OPEN_INT:
                diag["low_oi"] += 1;        continue
            iv = row.get("impliedVolatility")
            delta = bs_put_delta(px, K, dte / 365.0, float(iv) if iv else None)
            if delta is None or not (DELTA_MIN <= abs(delta) <= DELTA_MAX):
                diag["delta_out"] += 1
                if delta is not None: deltas_seen.append(abs(delta))
                continue
            y_ann = (mid / K) * (365.0 / dte)                  # premium / collateral, annualized
            c = {"ticker": sym, "px": px, "sma200": sma200, "gate_margin_%": (px / sma200 - 1) * 100,
                 "exp": exp, "dte": dte, "strike": K, "otm_%": (1 - K / px) * 100, "bid": bid, "ask": ask,
                 "mid": mid, "delta": round(delta, 3), "iv_%": round(float(iv) * 100, 1) if iv else None,
                 "oi": int(oi), "collateral_$": K * 100, "premium_$": round(mid * 100, 0),
                 "yield_ann_%": round(y_ann * 100, 1),
                 "earnings_d": edays if edays is not None else "unknown",
                 "flag": "; ".join([f for f in [
                     "PRICED-FOR-A-REASON" if y_ann > YIELD_WARN else "",
                     "SHORT-DATED (expires pre-earnings)" if short_dated else "",
                     "earnings-date-UNKNOWN — may span a report" if edays is None else ""] if f])}
            if best is None or c["yield_ann_%"] > best["yield_ann_%"]: best = c
        if best is None:
            parts = [f"{v} {k}" for k, v in diag.items() if k != "otm_strikes" and v > 0]
            near = f"; nearest |delta| {max(deltas_seen):.2f}" if deltas_seen else ""
            hint = " (all quotes dead — off-hours run? rerun in market hours)" if \
                   diag["no_live_quote"] > 0 and diag["no_live_quote"] >= diag["otm_strikes"] - diag["collateral"] else ""
            reject(sym, "3", f"no strike passed [{diag['otm_strikes']} OTM: " + ", ".join(parts) + f"]{near}{hint}")
            continue
        passers.append(best)
    except Exception as ex:
        errors.append({"ticker": sym, "error": str(ex)[:120]})
print(f"screened {len(UNIVERSE)}: {len(passers)} pass, {len(rejects)} rejected, {len(errors)} data errors")

In [ ]:
# Cell 6 — RESULTS: ranked survivors, then the audit trail (Law 8: every reject shows its law)
# Regime context first: the index's own 200-day (broad stand-down awareness, not a law yet)
try:
    spy = yf.Ticker("SPY").history(period="1y", auto_adjust=True)["Close"]
    spx_px, spx_sma = float(spy.iloc[-1]), float(spy.rolling(200).mean().iloc[-1])
    print(f"REGIME: SPY {spx_px:.0f} vs 200-day {spx_sma:.0f} -> {'ABOVE (harvest season)' if spx_px > spx_sma else 'BELOW - MARKET-WIDE STAND-DOWN, be picky or sit out'}\n")
except Exception as e:
    print("SPY regime check failed:", e)

if passers:
    df = pd.DataFrame(passers).sort_values("yield_ann_%", ascending=False).reset_index(drop=True)
    cols = ["ticker","px","strike","otm_%","exp","dte","mid","premium_$","collateral_$",
            "yield_ann_%","delta","iv_%","oi","gate_margin_%","earnings_d","flag"]
    print("=== CANDIDATES (best strike per name, ranked by annualized premium yield) ===")
    display(df[cols].round(2))
    print("\nRead it like this: yield is the RENT, delta is roughly the odds of assignment, the flag column")
    print("is the screen telling you when rent looks too good. Final law-3 question is yours alone:")
    print("WOULD YOU GENUINELY OWN 100 SHARES AT THAT STRIKE? If no -> it is not a candidate, whatever the yield.")
else:
    print("No survivors today. That is a valid output — the gate exists to keep you OUT sometimes.")

if rejects:
    print("\n=== REJECTED (which law killed it) ===")
    with pd.option_context("display.max_colwidth", 120):
        display(pd.DataFrame(rejects).sort_values(["law","ticker"]).reset_index(drop=True))

if ladder:
    print("\n=== LAW-4 LADDER (info only, NOT candidates): what a bigger capital box would buy ===")
    print("Collateral shown is at-spot; a 0.15-0.35-delta strike runs ~5-20% below that. For the")
    print("live-funding decision someday, not for today.")
    display(pd.DataFrame(sorted(ladder, key=lambda r: r["px"])).reset_index(drop=True))

if errors:
    print("\n=== DATA ERRORS (rerun later; yfinance hiccups) ===")
    display(pd.DataFrame(errors))

### After the run
1. Sanity-check 2–3 survivors by hand (chart, news, "would I own it here").
2. Paste the CANDIDATES table back into the session — we tune thresholds on real output, then wire the
   winner into the **paper** propose→validate loop (the risk engine re-checks every law before any order).
3. Off-hours caveat: quotes go stale after the close — treat yields as approximate until you rerun in market hours.

*Owner's own capital, self-directed. Not investment advice. The screen proposes; Jake disposes.*